# 01 · Teacher embeddings (inference only)
The teacher (Qwen3-Embedding-4B) is **already trained and stays frozen**: this is embedding generation, not training.
It embeds the corpus and every query set once (MRL d=256, fp16) on 2 GPUs. These vectors are the fixed document index
and the targets every later notebook trains against.

How a text becomes a vector (`model.encode`): queries get the instruction prompt, passages none → tokenize with left
padding, end-of-text token appended → forward → last-token pooling → first 256 dims (MRL) → L2 normalise.
Batches are length-sorted; each GPU embeds its own shard; shards already on disk are skipped (resumable).

Input: `00`. Output (attach to `02`, `03`, `04`): `cache/<mode>/{data.json, emb/teacher/*.npy}`, `runs/<mode>/metrics.json` (+ row `teacher`).

In [ ]:
# setup: clone/pull the repo on Kaggle, put it on sys.path, install what's missing (re-run to pull latest code)
import os, sys, importlib.util
REPO_URL = "https://github.com/pyther-hub/hlm-poc.git"
ON_KAGGLE = os.path.exists("/kaggle/working")
REPO = "/kaggle/working/hlm-poc" if ON_KAGGLE else os.path.abspath("..")
if ON_KAGGLE:
    if not os.path.exists(REPO):
        !git clone -q {REPO_URL} {REPO}
    else:
        !git -C {REPO} pull -q
sys.path.insert(0, REPO)

# HF token (higher rate limits, faster downloads). Kaggle: Add-ons -> Secrets -> add "HF_TOKEN", attach to notebook.
# Spawned workers inherit os.environ. Locally: export HF_TOKEN or `huggingface-cli login`.
if ON_KAGGLE and "HF_TOKEN" not in os.environ:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print(f"HF_TOKEN secret not found ({type(e).__name__}): downloads stay unauthenticated")

import transformers
from packaging.version import Version
if Version(transformers.__version__) < Version("4.56"):  # Qwen3 + the dtype= kwarg
    !pip install -q -U "transformers>=4.56"
    print("transformers upgraded: restart the kernel, then re-run this cell")
for pkg in ("datasets", "pyarrow", "accelerate"):
    if importlib.util.find_spec(pkg) is None:
        !pip install -q {pkg}

%load_ext autoreload
%autoreload 2

## Control panel

In [ ]:
import os
import shutil
import torch
from teacher import QUERY_PROMPT

DEBUG = True    # use the same value as in 00
NPROCS = 2

# --- paths. On Kaggle, earlier notebooks' outputs are attached as inputs (read only); set the slugs to yours.
#     Off Kaggle everything lives under the repo, so every IN_* is the same dir.
ROOT = "/kaggle/working" if ON_KAGGLE else REPO
MODE = "debug" if DEBUG else "full"
KAGGLE_IN = "/kaggle/input/notebooks/tensorpanda231"
IN_00 = f"{KAGGLE_IN}/hlm-00-data-audit" if ON_KAGGLE else REPO
CACHE_DIR = f"{ROOT}/cache/{MODE}"
DATA_PATH = f"{CACHE_DIR}/data.json"
EMB_DIR = f"{CACHE_DIR}/emb/teacher"
RUN_DIR = f"{ROOT}/runs/{MODE}"
PREV_METRICS = f"{IN_00}/runs/{MODE}/metrics.json"

TEACHER = dict(model="Qwen/Qwen3-Embedding-4B", dim=256, dtype="fp16",
               query_prompt=QUERY_PROMPT,          # queries only; passages get no prefix
               doc_max_len=128, query_max_len=64,  # tokens; query length includes the ~20-token prompt
               batch_size=64,                      # per GPU
               sets=("corpus", "align", "val", "test"), data_path=DATA_PATH, emb_dir=EMB_DIR)
CHECKS = dict(sanity_n=1000, timing_n=2000, budget_h=5.0)   # fp16 check / timing sample / reported budget

## 1 · Inputs
`data.json` is copied from `00`, so later notebooks only need this notebook's output.

In [ ]:
from data import load_json

os.makedirs(CACHE_DIR, exist_ok=True)
if not os.path.exists(DATA_PATH):
    shutil.copy(f"{IN_00}/cache/{MODE}/data.json", DATA_PATH)
# rows accumulate across notebooks: start from the previous notebook's metrics.json (copied once)
os.makedirs(RUN_DIR, exist_ok=True)
if not os.path.exists(f"{RUN_DIR}/metrics.json") and os.path.exists(PREV_METRICS):
    shutil.copy(PREV_METRICS, f"{RUN_DIR}/metrics.json")
data = load_json(DATA_PATH)
print({k: len(v["ids"]) for k, v in data.items()})

## 2 · Pre-flight
fp16 vs fp32 on a passage sample (must have no NaNs), and a projection of how long the full run takes.

In [ ]:
from model import get_device
from teacher import fp16_sanity, free_gpu_memory, time_embedding

device = get_device()
if os.path.exists(f"{EMB_DIR}/corpus.npy"):
    print("corpus already embedded, pre-flight skipped")
else:
    sanity = fp16_sanity(TEACHER["model"], data["corpus"]["texts"][:CHECKS["sanity_n"]], TEACHER["dim"],
                         TEACHER["doc_max_len"], device, device_map_fp32="auto" if torch.cuda.device_count() > 1 else None)
    assert sanity["nan_rows"] == 0, "fp16 teacher produced NaNs"
    time_embedding(TEACHER, data, device, world_size=NPROCS, n_sample=CHECKS["timing_n"], budget_h=CHECKS["budget_h"])
    free_gpu_memory()

## 3 · Embed on both GPUs
Rank r embeds shard r of every set; rank 0 merges the shards into `<set>.npy` + `<set>_ids.json`.

In [ ]:
import time
from ddp import launch
from teacher import embed_main

t0 = time.time()
launch(embed_main, TEACHER, nprocs=NPROCS)
n_texts = sum(len(data[s]["ids"]) for s in TEACHER["sets"])
print(f"{n_texts} texts in {time.time() - t0:.0f} s ({n_texts / (time.time() - t0):.0f} texts/s, ~0 if already done)")

## 4 · Validate
Every set: shape `(N, 256)`, finite, unit norm, ids exactly `data.json`'s (so no record is missing or reordered).

In [ ]:
import numpy as np
from data import load_emb
from evaluation import check_emb

for name in TEACHER["sets"]:
    emb, ids = load_emb(EMB_DIR, name)
    assert emb.shape == (len(data[name]["ids"]), TEACHER["dim"]), f"{name}: shape {emb.shape}"
    check_emb(emb, ids, data[name]["ids"], name)   # finite, norms ~1, ids in data.json order
    norms = np.linalg.norm(emb.astype(np.float32), axis=1)
    print(f"{name:<7} {str(emb.shape):<15} {emb.dtype}  norm {norms.min():.4f}..{norms.max():.4f}  "
          f"{os.path.getsize(f'{EMB_DIR}/{name}.npy') / 1e6:8.1f} MB")

## 5 · Teacher ceiling
Teacher queries vs teacher corpus: the best any student can reach on this index. Row `teacher` in `metrics.json`.

In [ ]:
from evaluation import load_index, save_metrics, teacher_ceiling

docs, teacher_q = load_index(data, EMB_DIR, ("val", "test"))
rows = teacher_ceiling(data, docs, teacher_q, 100, device)
save_metrics(RUN_DIR, rows)
for split, m in rows["teacher"].items():
    print(split, {k: round(v, 4) for k, v in m.items()})